# 02 · Dimensionality reduction and clustering
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/en/02_dimensionality_reduction_clustering.ipynb)

**From Cells to Clusters: An Introduction to Single-Cell RNA-seq Analysis in Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Peru · November 9, 2026  
Instructor: Dr. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Time in the tutorial:** 10:15–10:45 (30 min)

**By the end of this notebook you will be able to:**
- Run PCA and choose how many principal components to keep
- Build a k-nearest-neighbor graph and a UMAP embedding (and know what UMAP does *not* tell you)
- Cluster cells with the Leiden algorithm and explore the effect of resolution
- Check clusters for technical artifacts

> 🟢 **Core** sections are covered live. 🔎 **Extra** sections are for self-study after the tutorial.

## 0. Setup 🟢
Run the cell below first (▶ or `Shift + Enter`). On Google Colab it installs the packages (~1 min);
on your own computer it assumes you already ran `pip install -r requirements.txt`.

**Optional:** set `USE_DRIVE = True` to save your results in Google Drive, so the next notebooks
can reuse them. If you leave it `False`, each notebook rebuilds what it needs automatically.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10"

USE_DRIVE = False  # True = keep results in Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Tutorial helper functions (download + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # running from a local clone
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

## 1. Load the QC'd data 🟢
`tut.get_checkpoint("qc")` loads the file saved at the end of notebook 01. If it is not there (new Colab session),
it rebuilds it from the raw data with the same parameters.

In [ ]:
adata = tut.get_checkpoint("qc", WORKDIR)
adata

## 2. PCA: from 2,000 genes to ~30 axes 🟢
Genes are correlated (a T cell expresses *CD3D*, *CD3E*, *IL7R* together), so a few **principal components**
capture most of the biological signal. Downstream steps use the PCs instead of genes: faster and less noisy.

> 🔎 Many tutorials scale each gene to unit variance (`sc.pp.scale`) before PCA. Current best practice
> (sc-best-practices.org) often skips it; both work for PBMCs.

In [ ]:
sc.tl.pca(adata, n_comps=50, mask_var="highly_variable", random_state=0)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

**Elbow plot:** after the "elbow" each extra PC adds little. For PBMC 3k it is around PC 10–15; keeping
a few more (we use **30**) is safe, while keeping too few can merge rare populations.

In [ ]:
sc.pl.pca(adata, color=["CST3", "NKG7", "MS4A1", "pct_counts_mt"], ncols=4, use_raw=True)

PC1 already separates myeloid cells (*CST3*) from lymphocytes, and % MT is not driving the main axes: good sign.

## 3. Neighbor graph and UMAP 🟢
1. **kNN graph:** each cell is connected to its 15 most similar cells in PCA space. Clustering and UMAP both use this graph.
2. **UMAP** draws that graph in 2D for visualization.

⚠️ UMAP preserves **local** neighborhoods. Distances **between** clusters and cluster **sizes** in a UMAP are not
reliable: never draw biological conclusions from them alone.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts", "pct_counts_mt"], ncols=3)

In [ ]:
sc.pl.umap(adata, color=["CD3E", "CD8A", "GNLY", "MS4A1", "CD14", "FCGR3A", "FCER1A", "PPBP"],
           ncols=4, use_raw=True, cmap="viridis")

## 4. Leiden clustering 🟢
Leiden finds groups of cells that are more connected to each other than to the rest of the graph.
The **resolution** controls how many clusters you get: higher = more, smaller clusters.
There is no single correct value; it depends on the biological question.

In [ ]:
for res in [0.3, 0.8, 1.5]:
    sc.tl.leiden(adata, resolution=res, key_added=f"leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)
    print(f"resolution {res}: {adata.obs[f'leiden_{res}'].nunique()} clusters")

sc.pl.umap(adata, color=["leiden_0.3", "leiden_0.8", "leiden_1.5"], legend_loc="on data", ncols=3)

How do clusters split as resolution increases? A cross-table shows it (the R package **clustree** draws the same idea as a tree).

In [ ]:
pd.crosstab(adata.obs["leiden_0.3"], adata.obs["leiden_0.8"])

We keep **resolution 0.8** as our main clustering (`adata.obs["leiden"]`).

## 5. Are any clusters technical artifacts? 🟢
A cluster defined only by low UMIs or high % MT is usually low-quality cells, not a cell type.

In [ ]:
adata.obs["leiden"] = adata.obs["leiden_0.8"]
sc.pl.violin(adata, ["n_genes_by_counts", "pct_counts_mt", "doublet_score"], groupby="leiden",
             stripplot=False, multi_panel=True)
adata.obs["leiden"].value_counts().sort_index().to_frame("cells").T

## 6. 🔎 Extra: t-SNE and PAGA
t-SNE is an older alternative to UMAP. PAGA summarizes how clusters connect, useful for continuous processes
(differentiation) and as a better initialization for UMAP.

In [ ]:
sc.tl.tsne(adata, n_pcs=30, random_state=0)
sc.tl.paga(adata, groups="leiden")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sc.pl.tsne(adata, color="leiden", ax=axes[0], show=False, legend_loc="on data")
sc.pl.paga(adata, ax=axes[1], show=False, fontsize=9)
plt.tight_layout()

## 7. Save the checkpoint 🟢

In [ ]:
adata.write_h5ad(f"{WORKDIR}/results/pbmc3k_clustered.h5ad")

### 💡 Your turn
Recompute the neighbor graph with `n_neighbors=5` and with `n_neighbors=50`, run UMAP each time and compare
with the plot above (color by `leiden`). What changes? Do the clusters change?

In [ ]:
# Your code here

<details><summary><b>Show solution</b></summary>

```python
for k in (5, 50):
    tmp = adata.copy()
    sc.pp.neighbors(tmp, n_neighbors=k, n_pcs=30, random_state=0)
    sc.tl.umap(tmp, random_state=0)
    sc.pl.umap(tmp, color="leiden", title=f"n_neighbors = {k}", legend_loc="on data")
# Small k: fragmented, local structure. Large k: smoother, more global structure.
# Same colors = same clusters (computed before): UMAP is only a picture of the graph.
```
</details>

---
## ✅ Done!
You have clusters. But what cell types are they? Next: **03 · Cell type annotation**.